<a href="https://colab.research.google.com/github/arunkumar7786/gen-AI/blob/main/Module5_Source_Cited_Research_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [ ]:
# ! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

In [ ]:
# OpenRouter Environment
import os
from getpass import getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('Enter your OpenRouter API key: ')

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_MODEL = 'openai/gpt-4o-mini'

# OpenRouter is OpenAI-compatible, but uses its own API key and endpoint.


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [ ]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from ddgs import DDGS  # type: ignore
except Exception:
    try:
        from duckduckgo_search import DDGS  # type: ignore
    except Exception:
        DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [ ]:
# Q1 — Solution: DuckDuckGo search
def ddg_search(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    """Return normalized search results. Network/API failures return an empty list."""
    if not query or not str(query).strip():
        return []
    rows: List[Dict[str, Any]] = []
    try:
        if DDGS is None:
            return rows
        with DDGS() as ddgs:
            try:
                raw = ddgs.text(str(query), max_results=max(1, min(int(max_results), 20)))
            except TypeError:
                raw = ddgs.text(str(query))
            for item in raw or []:
                url = item.get("href") or item.get("url") or ""
                if not url.startswith(("http://", "https://")):
                    continue
                rows.append({
                    "title": str(item.get("title") or url),
                    "url": url,
                    "snippet": str(item.get("body") or item.get("snippet") or ""),
                    "source": "DuckDuckGo"
                })
                if len(rows) >= max_results:
                    break
    except Exception as exc:
        print(f"DuckDuckGo search unavailable: {exc}")
    return rows


In [ ]:
def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Retrieve a concise summary from the Wikipedia REST API."""
    fallback = {"title": str(title_or_query), "url": "", "summary": ""}
    if requests is None:
        return fallback
    try:
        slug = str(title_or_query).strip().replace(" ", "_")
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(slug, safe='()_')}"
        response = requests.get(url, headers={"User-Agent": "ResearchBriefAgent/1.0"}, timeout=15)
        if response.status_code == 404:
            search = requests.get("https://en.wikipedia.org/w/api.php",
                params={"action":"query","list":"search","srsearch":title_or_query,"format":"json"},
                headers={"User-Agent":"ResearchBriefAgent/1.0"}, timeout=15)
            search.raise_for_status()
            hits = search.json().get("query", {}).get("search", [])
            if not hits:
                return fallback
            slug = hits[0]["title"].replace(" ", "_")
            url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(slug, safe='()_')}"
            response = requests.get(url, headers={"User-Agent":"ResearchBriefAgent/1.0"}, timeout=15)
        response.raise_for_status()
        data = response.json()
        summary = str(data.get("extract", ""))
        parts = re.split(r'(?<=[.!?])\s+', summary)
        summary = " ".join(parts[:max(1, int(sentences))])
        return {"title": str(data.get("title") or title_or_query),
                "url": str(data.get("content_urls", {}).get("desktop", {}).get("page") or url),
                "summary": summary}
    except Exception as exc:
        print(f"Wikipedia lookup unavailable: {exc}")
        return fallback


In [ ]:
def fetch_page_clean(url: str) -> Dict[str, str]:
    """Fetch a public HTTP(S) page and extract readable text while removing HTML noise."""
    result = {"title": "", "url": str(url), "text": ""}
    if requests is None or BeautifulSoup is None:
        return result
    try:
        parsed = urlparse(str(url))
        if parsed.scheme not in ("http", "https") or not parsed.netloc:
            return result
        response = requests.get(str(url), headers={"User-Agent":"Mozilla/5.0 (compatible; ResearchBriefAgent/1.0)"},
                                timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        title = soup.title.get_text(" ", strip=True) if soup.title else parsed.netloc
        for node in soup(["script", "style", "noscript", "nav", "footer", "header", "aside", "form", "svg"]):
            node.decompose()
        main = soup.find("main") or soup.find("article") or soup.body or soup
        text = " ".join(main.get_text(" ", strip=True).split())
        return {"title": title, "url": str(response.url), "text": text[:18000]}
    except Exception as exc:
        print(f"Could not fetch {url}: {exc}")
        result["error"] = str(exc)
        return result


In [ ]:
def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    """Rank URLs by usable metadata and domain diversity; return unique HTTP(S) URLs."""
    candidates = []
    seen, domains = set(), set()
    for item in results or []:
        url = str(item.get("url") or item.get("href") or "")
        parsed = urlparse(url)
        if parsed.scheme not in ("http", "https") or not parsed.netloc or url in seen:
            continue
        seen.add(url)
        domain = parsed.netloc.lower().removeprefix("www.")
        title = str(item.get("title") or "")
        snippet = str(item.get("snippet") or item.get("body") or "")
        score = min(len(snippet), 500) / 100 + (2 if title else 0)
        candidates.append((score, domain, url))
    candidates.sort(key=lambda x: x[0], reverse=True)
    chosen = []
    # First pass favors distinct domains; second pass fills remaining slots.
    for _, domain, url in candidates:
        if domain not in domains and len(chosen) < max(1, k):
            chosen.append(url); domains.add(domain)
    for _, domain, url in candidates:
        if len(chosen) >= max(1, k): break
        if url not in chosen: chosen.append(url)
    return chosen


In [ ]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
    """Persist JSON logs, creating parent directories as needed."""
    target = pathlib.Path(path)
    if not target.is_absolute():
        target = DATA_DIR / target
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return str(target)


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

In [ ]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [ ]:
# Q2 — Solution: PydanticAI agent and registered tools
SYSTEM_INSTRUCTIONS = """
You are a careful web-research assistant using a ReAct workflow.
For current, factual, disputed, or specialized topics, search DuckDuckGo first.
Use Wikipedia for concise background or disambiguation, not as the only authority.
Fetch promising pages with fetch_page_clean; use select_top_urls to prioritize sources.
Use save_logs to persist search results, selected URLs, and snippets.
Base claims only on retrieved tool output. Never invent facts, URLs, quotations, or publication dates.
When drafting, cite sources inline as [1], [2], etc. and provide a References list.
Use at least three distinct domains when available. Quote exact source wording only,
keep each quotation at ten words or fewer, and attribute it with a numeric citation.
If evidence is missing or sources disagree, state the limitation clearly.
"""

if Agent is None or OPENROUTER_MODEL_INSTANCE is None:
    raise RuntimeError("PydanticAI/OpenRouter model unavailable. Install dependencies and set OPENROUTER_API_KEY.")

agent = Agent(OPENROUTER_MODEL_INSTANCE, instructions=SYSTEM_INSTRUCTIONS)
_TOOL_LOGS: Dict[str, Any] = {"search_results": [], "wiki": [], "pages": [], "selected_urls": []}

@agent.tool_plain
def ddg_search_tool(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    """Search the web with DuckDuckGo and return titles, URLs, and snippets."""
    rows = ddg_search(query, max_results)
    _TOOL_LOGS["search_results"].extend(rows)
    return rows

@agent.tool_plain
def wiki_summary_tool(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Retrieve a short Wikipedia REST API summary for background context."""
    item = wiki_summary(title_or_query, sentences)
    _TOOL_LOGS["wiki"].append(item)
    return item

@agent.tool_plain
def fetch_page_clean_tool(url: str) -> Dict[str, str]:
    """Fetch a web page and extract cleaned readable text."""
    item = fetch_page_clean(url)
    _TOOL_LOGS["pages"].append(item)
    return item

@agent.tool_plain
def select_top_urls_tool(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    """Select useful, unique URLs while preferring domain diversity."""
    urls = select_top_urls(results, k)
    _TOOL_LOGS["selected_urls"] = list(dict.fromkeys(_TOOL_LOGS.get("selected_urls", []) + urls))
    return urls

@agent.tool_plain
def save_logs_tool(payload: Dict[str, Any], path: str = "artifacts/tool_logs.json") -> str:
    """Save research tool logs as JSON in the artifacts folder."""
    return save_logs(payload, path)


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [ ]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [ ]:
# Q3 — Solution: conversational ReAct runs with retained message history
import os, asyncio, inspect
from typing import Any, Dict

def run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]:
    if not str(topic).strip():
        raise ValueError("Please provide a research topic.")
    _TOOL_LOGS.clear()
    _TOOL_LOGS.update({"search_results": [], "wiki": [], "pages": [], "selected_urls": []})
    prompt = (
        f"Research topic: {topic}\n"
        "Create a research draft supported by web sources. Search broadly, consult Wikipedia "
        "when useful, fetch several credible pages, and save your tool logs. Identify important "
        "findings, caveats, and areas of disagreement. Do not invent citations or quotations."
    )
    result = agent.run_sync(prompt)
    history = result.new_messages()
    drafts = [str(result.output)]
    # Optional interactive follow-up turns. Each turn reuses PydanticAI message history.
    for turn in range(max(1, int(rounds)) - 1):
        try:
            question = input(f"Follow-up question {turn + 1} (press Enter to finish): ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not question:
            break
        follow = agent.run_sync(question, message_history=history)
        history = history + follow.new_messages()
        drafts.append(str(follow.output))
    # Deterministic retrieval fallback: if the model elects not to search enough,
    # collect a small evidence set so the deliverable still records its sources.
    if len(_TOOL_LOGS["search_results"]) < 5:
        extra = ddg_search(str(topic), max_results=10)
        known = {x.get("url") for x in _TOOL_LOGS["search_results"]}
        _TOOL_LOGS["search_results"].extend(x for x in extra if x.get("url") not in known)
    preliminary = select_top_urls(_TOOL_LOGS["search_results"], k=8)
    for url in preliminary:
        if url not in _TOOL_LOGS["selected_urls"]:
            _TOOL_LOGS["selected_urls"].append(url)
        if len(_TOOL_LOGS["pages"]) < 8 and not any(p.get("url") == url for p in _TOOL_LOGS["pages"]):
            _TOOL_LOGS["pages"].append(fetch_page_clean(url))
    if not _TOOL_LOGS["wiki"]:
        wiki_item = wiki_summary(str(topic), sentences=3)
        if wiki_item.get("summary"):
            _TOOL_LOGS["wiki"].append(wiki_item)
    pages = [p for p in _TOOL_LOGS["pages"] if p.get("text")]
    wiki_items = [w for w in _TOOL_LOGS["wiki"] if w.get("summary")]
    # Keep search results plus fetched page metadata for traceability.
    all_sources = []
    for item in _TOOL_LOGS["search_results"]:
        all_sources.append({"title": item.get("title",""), "url": item.get("url",""),
                            "snippet": item.get("snippet",""), "source":"DuckDuckGo"})
    for item in wiki_items:
        all_sources.append({"title": item["title"], "url": item["url"],
                            "snippet": item["summary"], "source":"Wikipedia"})
    for item in pages:
        all_sources.append({"title": item.get("title",""), "url": item.get("url",""),
                            "snippet": item.get("text","")[:2500], "source":"Fetched page"})
    # Ensure URLs selected by the agent are represented even if a fetch failed.
    chosen = list(dict.fromkeys(_TOOL_LOGS["selected_urls"]))
    if not chosen:
        chosen = select_top_urls(all_sources, k=8)
    _TOOL_LOGS["selected_urls"] = chosen
    report = {"topic": topic, "draft": "\n\n".join(drafts), "messages": history,
              "search_results": _TOOL_LOGS["search_results"], "wiki": wiki_items,
              "pages": pages, "sources": all_sources, "selected_urls": chosen,
              "snippets": [s for s in all_sources if s.get("snippet")]}
    save_logs(_TOOL_LOGS, str(ARTIFACTS_DIR / "tool_logs.json"))
    save_logs(report, str(ARTIFACTS_DIR / "report_raw.json"))
    return report


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

In [ ]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [ ]:
# Q4 — Solution: grounded synthesis, numeric citations, short exact quotations
def synthesize_brief(report: Dict[str, Any], min_paras: int = 5, max_paras: int = 8) -> Dict[str, Any]:
    topic = str(report.get("topic") or "Research topic")
    raw_sources = report.get("sources") or report.get("snippets") or []
    selected = set(report.get("selected_urls") or [])
    sources, seen = [], set()
    for item in raw_sources:
        if not isinstance(item, dict):
            continue
        url = str(item.get("url") or "")
        if not url.startswith(("http://", "https://")) or url in seen:
            continue
        if selected and url not in selected and len(sources) >= 8:
            continue
        seen.add(url)
        sources.append({"title": str(item.get("title") or urlparse(url).netloc),
                        "url": url, "snippet": str(item.get("snippet") or item.get("text") or "")})
    if not sources:
        for url in report.get("selected_urls", []):
            if str(url).startswith(("http://", "https://")) and url not in seen:
                seen.add(url); sources.append({"title": urlparse(url).netloc, "url":url, "snippet":""})
    sources = sources[:10]
    references = [{"number": i+1, "title": s["title"], "url": s["url"]} for i,s in enumerate(sources)]

    # Extract two short, verbatim phrases from retrieved content (never manufacture quotations).
    quotes = []
    for source in sources:
        text_value = re.sub(r'\s+', ' ', source["snippet"]).strip()
        for sentence in re.split(r'(?<=[.!?])\s+', text_value):
            words = re.findall(r"[\w’'-]+", sentence)
            if 3 <= len(words) <= 10:
                phrase = " ".join(words)
                if phrase not in quotes:
                    quotes.append(phrase)
                break
        if len(quotes) >= 2:
            break

    # The agent's draft is the model synthesis; split it into readable paragraphs.
    draft = re.sub(r'\s+', ' ', str(report.get("draft") or "")).strip()
    sentences = re.split(r'(?<=[.!?])\s+', draft) if draft else []
    paras = []
    if sentences:
        target = max(min_paras, min(max_paras, 5))
        chunks = [[] for _ in range(target)]
        for idx, sentence in enumerate(sentences):
            chunks[min(idx * target // max(1, len(sentences)), target-1)].append(sentence)
        paras = [" ".join(chunk).strip() for chunk in chunks if chunk]
    # Build five transparent, source-grounded paragraphs when the draft is too short.
    source_lines = []
    for i,s in enumerate(sources):
        snippet = re.sub(r'\s+', ' ', s["snippet"]).strip()
        if snippet:
            source_lines.append((i+1, s["title"], snippet[:650]))
    if len(paras) < min_paras:
        paras = [f"This brief examines **{topic}** using retrieved web material rather than relying on unsupported background assumptions. The available evidence is summarized below."]
        if source_lines:
            for number,title,snippet in source_lines[:3]:
                paras.append(f"**Source finding:** {title} reports the following relevant material: {snippet} [{number}].")
        while len(paras) < min_paras:
            paras.append("The retrieved material provides a starting point for understanding the topic; conclusions should be limited to what the cited pages actually establish.")
    paras = paras[:max_paras]
    # Attach numeric citations to draft paragraphs when sources exist.
    if references:
        for i in range(len(paras)):
            if not re.search(r'\[\d+\]', paras[i]):
                paras[i] = paras[i].rstrip() + f" [{(i % len(references))+1}]"
    else:
        paras = [p + " (No retrievable source was available to substantiate this point.)" for p in paras]

    # Insert exact short quotes from source snippets and cite their origin.
    if len(quotes) >= 1 and references:
        paras[0] += f' One retrieved source states, “{quotes[0]}” [1].'
    if len(quotes) >= 2 and references:
        quote_ref = min(2, len(references))
        paras[min(1, len(paras)-1)] += f' Another source describes it as, “{quotes[1]}” [{quote_ref}].'
    elif len(references):
        # Explicitly disclose when the retrieved pages do not provide two short extractable quotations.
        paras[-1] += " The fetched text did not contain a second short, safely extractable quotation."

    ref_lines = ["## References"]
    for ref in references:
        ref_lines.append(f'[{ref["number"]}] [{ref["title"]}]({ref["url"]})')
    brief = "\n\n".join(paras + ["\n".join(ref_lines)])
    report["brief"] = brief
    report["references"] = references
    report["quotes"] = quotes[:2]
    (ARTIFACTS_DIR / "brief.md").write_text(brief, encoding="utf-8")
    return report


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

In [ ]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [ ]:
# Q5 — Solution: end-to-end pipeline
def run_pipeline(topic: str) -> Dict[str, Any]:
    report = run_agent(topic)
    report = synthesize_brief(report)
    # Persist the complete report, including the final brief and references.
    save_logs(report, str(ARTIFACTS_DIR / "report.json"))
    return report


In [ ]:
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'),
                    report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
    brief_file, report_file = pathlib.Path(brief_path), pathlib.Path(report_path)
    brief = brief_file.read_text(encoding="utf-8") if brief_file.exists() else ""
    report = json.loads(report_file.read_text(encoding="utf-8")) if report_file.exists() else {}
    refs = report.get("references", [])
    domains = {urlparse(str(r.get("url",""))).netloc.lower().removeprefix("www.")
               for r in refs if r.get("url")}
    # Count quoted spans and verify each has at most ten words.
    quoted = re.findall(r'[“"]([^”"]+)[”"]', brief)
    short_quotes = [q for q in quoted if len(re.findall(r"[\w’'-]+", q)) <= 10]
    logs_exist = (ARTIFACTS_DIR / "tool_logs.json").exists() and report_file.exists()
    scorecard = {
        "unique_domains": len(domains),
        "has_three_domains": len(domains) >= 3,
        "quote_count": len(short_quotes),
        "has_two_short_quotes": len(short_quotes) >= 2,
        "has_references": "## References" in brief or "References" in brief,
        "logs_exist": logs_exist,
        "brief_exists": brief_file.exists(),
        "report_exists": report_file.exists(),
        "passed": len(domains) >= 3 and len(short_quotes) >= 2 and
                  ("References" in brief) and logs_exist
    }
    save_logs(scorecard, str(ARTIFACTS_DIR / "scorecard.json"))
    return scorecard


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

In [ ]:
# Q5 — Hidden Tests

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).

## Reflection (500–700 words)

# Reflection: Building a Source-Cited Web Research Agent

Grounding an AI agent with external sources improves the reliability and usefulness of its answers because the model can base its response on information retrieved for the actual question. In this assignment, DuckDuckGo provides a way to discover current web pages, Wikipedia supplies quick background context, and page fetching provides text that can be inspected before writing. This division of work helps reduce dependence on the model's stored knowledge, which may be incomplete or outdated. Numeric citations and a References section also make it easier for a reader to verify important claims. However, grounding does not guarantee truth. Search engines can return low-quality or promotional pages, Wikipedia may simplify complex topics, and even credible sources can be outdated or disagree. The agent therefore needs to describe uncertainty and avoid presenting a retrieved statement as universally true.

Tool use also introduces cost and latency. A simple question answered directly by a model may require only one model request. A research task can involve a search request, several page-fetch requests, a Wikipedia request, and one or more model calls. Each additional operation increases execution time and may introduce a failure point. Fetching too many pages can make the process slow and produce more text than the model can use efficiently. On the other hand, fetching too few pages can leave the brief poorly supported. A practical design should limit the number of search results and pages, prefer distinct domains, and stop retrieving once enough relevant evidence has been collected. Caching previously fetched pages and search results would reduce repeated network requests and improve response time for recurring topics.

Prompting has a major effect on the quality and reliability of the output. The system instructions in this project tell the agent when to search, when Wikipedia is useful, and why pages should be fetched before synthesis. They also prohibit invented URLs, unsupported claims, and fabricated quotations. Explicit citation instructions encourage the model to connect claims to evidence rather than append references randomly. Still, prompts are not hard guarantees. A model may overlook a tool, cite the wrong source, or produce a quotation that is not present in the retrieved text. For that reason, the pipeline stores tool results and source snippets separately and applies basic checks to the final brief. Stronger implementations should validate every citation number against the reference list and verify each quotation against the original extracted text.

Extracting useful content from HTML is another challenge. Modern websites contain navigation menus, cookie notices, advertisements, scripts, footers, related-article links, and other elements that are not part of the main article. BeautifulSoup makes it possible to remove common noisy elements and extract text from an article or main content container. However, websites do not follow one consistent HTML structure. Some pages place the article in unusual containers, load content dynamically with JavaScript, or block automated requests. Removing too much HTML can discard useful tables, captions, or headings, while removing too little leaves noisy text that distracts the model. The current cleaner uses common tags and a preference for main or article elements, but it cannot handle every website. A production version could use readability-based extraction, site-specific parsers, and a fallback browser renderer for JavaScript-heavy pages.

There are several ways to extend this system. First, a reranker could score search results for relevance to the original question before pages are fetched. This would reduce unnecessary requests and improve source quality. Second, caching could store normalized URLs, fetch timestamps, and extracted text so repeated research does not download the same pages. Third, source quality controls could prefer primary research, government publications, official documentation, and established publishers while flagging anonymous or highly promotional pages. Safety controls should block private-network URLs, restrict redirects, enforce response-size and timeout limits, and avoid downloading executable content. The system could also record retrieval timestamps, validate quote spans, detect duplicate sources, and ask the user for clarification when the research question is too broad. Overall, the assignment demonstrates that a useful research agent is not simply a language model with search access: it is a coordinated workflow that retrieves evidence, tracks provenance, communicates uncertainty, and checks its own outputs.


## Final run and submission checklist

1. Run the notebook from top to bottom in Google Colab with your OpenRouter key entered only at the secure prompt.
2. In the final pipeline cell, replace the example topic with your chosen research topic and run `report = run_pipeline(topic)`.
3. Run `scorecard = evaluate_outputs()` and inspect `artifacts/scorecard.json`. If a check fails, broaden the topic/search, rerun, and inspect the source log.
4. Download the `artifacts/` folder and submit it with this notebook. Never commit your API key.


In [ ]:
# Final execution cell — edit the topic, then run after completing all setup cells.
RESEARCH_TOPIC = "The impact of generative AI on education: opportunities, risks, and evidence"
# report = run_pipeline(RESEARCH_TOPIC)
# scorecard = evaluate_outputs()
# print(json.dumps(scorecard, indent=2))
